In [2]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
import prism


In [3]:
# import pandas as pd
# import xarray as xr
# from imagematerials.util import dataset_to_array
# maintenance_fp = Path("../data/raw/vehicles/standard_data/maintenance_passenger_cars.csv")
# df = pd.read_csv(maintenance_fp, index_col=0)
# xr.DataArray(df["total_material_per_km"], dims=("Material",), coords={"Material": df["Material"]})
# print(df)
# dataset_to_array(df.to_xarray(), ["Material"], [])


In [4]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema2.nc")

In [5]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [6]:
prep_data["maintenance_material_fractions"]

<xarray.DataArray 'maintenance_material_fractions' (material: 14, Type: 53)> Size: 6kB
array([[0.00000000e+00,            nan, 2.92255640e-05, 2.92255640e-05,
        2.92255640e-05, 2.92255640e-05, 2.92255640e-05, 2.92255640e-05,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00,            nan, 4.35327334e-05, 4.35327334e-05,
        4.35327334e-05, 4.35327334e-05, 4.35327334e-05, 4.35327334e-05,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
...
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00,            nan, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00]])
Coordinates:
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'
  * Type      (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'

In [7]:
import xarray as xr
import numpy as np
import pandas as pd

material_fractions = prep_data['material_fractions']

maintenance_material = pd.read_csv("../data/raw/vehicles/standard_data/maintenance_passenger_cars.csv", index_col=0)


# Assuming material_fractions has the needed dimensions
materials = material_fractions.coords["material"]
types = material_fractions.coords["Type"]

# Initialize xr_maintenance_material with zeros
xr_maintenance_material = xr.DataArray(
    np.zeros((len(materials), len(types))),  # Shape based on dimensions
    dims=("material", "Type"),
    coords={"material": material_fractions.coords["material"], "Type": material_fractions.coords["Type"]}
)

# Assign values from data in xr_maintenance_material where Type contains "Cars"
cars_mask = np.char.find(types.astype(str), "Cars") >= 0  # Find entries containing "Cars"
xr_maintenance_material.loc[dict(Type=types[cars_mask])] = maintenance_material["total_material_per_km"].values[:, np.newaxis]

xr_maintenance_material.sel(Type="Bikes")


<xarray.DataArray (material: 14)> Size: 112B
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Coordinates:
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'
    Type      <U35 140B 'Bikes'

In [8]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [9]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [10]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

TypeError: GenericMaterials.__init__() got an unexpected keyword argument 'maintenance_material_fractions'

In [ ]:
main_model_factory.simulate(simulation_timeline)

In [11]:
main_model_normal.simulate(simulation_timeline)

IndexError: index 1961 is out of bounds for axis 0 with size 101

In [ ]:
assert(main_model_factory.stock_by_cohort_maintenance_materials == main_model_normal.maintenance_model.stock_by_cohort_maintenance_materials).all().all()

In [17]:
main_model_normal.maintenance_model.weights.sel(Cohort=2058).drop_vars("Cohort")

<xarray.DataArray 'vehicle_weights' (Type: 53)> Size: 424B
array([1.64800000e+01,            nan, 1.61257000e+03, 1.05203000e+03,
       1.33567000e+03, 1.30319000e+03, 1.55917000e+03, 0.00000000e+00,
       9.18312970e+04, 1.69064329e+06,            nan, 1.46112000e+04,
       1.44377900e+04, 1.45889500e+04, 1.47495000e+04, 1.45489100e+04,
       0.00000000e+00, 4.06252623e+05, 3.08522039e+05,            nan,
                  nan, 1.58325000e+03, 1.56446000e+03, 1.58084000e+03,
       1.59824000e+03, 1.57650000e+03, 0.00000000e+00,            nan,
       7.53970000e+03, 7.45021000e+03, 7.52822000e+03, 7.61107000e+03,
       7.50755000e+03, 0.00000000e+00,            nan,            nan,
       6.71051000e+03, 6.63086000e+03, 6.70029000e+03, 6.77402000e+03,
       6.68190000e+03, 0.00000000e+00, 5.80232220e+04,            nan,
       1.36106700e+04, 1.34491300e+04, 1.35899400e+04, 1.37395000e+04,
       1.35526400e+04, 1.36135700e+04,            nan, 2.41452031e+05,
                  nan])
Coordinates:
  * Type     (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'

In [ ]:
main_model_normal.maintenance_model.maintenance_material_fractions

<xarray.DataArray 'maintenance_material_fractions' (material: 14, Type: 53)> Size: 6kB
array([[0.00000000e+00,            nan, 2.92255640e-05, 2.92255640e-05,
        2.92255640e-05, 2.92255640e-05, 2.92255640e-05, 2.92255640e-05,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00,            nan, 4.35327334e-05, 4.35327334e-05,
        4.35327334e-05, 4.35327334e-05, 4.35327334e-05, 4.35327334e-05,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
...
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00,            nan, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00,            nan, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                   nan, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00,            nan,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00]])
Coordinates:
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'
  * Type      (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'

In [18]:
main_model_normal.stock_model.stock_by_cohort

<xarray.DataArray (Time: 101, Cohort: 101, Region: 28, Type: 53)> Size: 121MB
array([[[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]]]])
Coordinates:
  * Time     (Time) int64 808B 1960 1961 1962 1963 1964 ... 2057 2058 2059 2060
  * Cohort   (Cohort) int64 808B 1960 1961 1962 1963 ... 2057 2058 2059 2060
  * Region   (Region) <U2 224B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
  * Type     (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'